# Run OpenTRIM (`opentrim`) from a Python notebook

This cell runs the `opentrim` executable with after generating a custom json input file. It also prints:
- the exit code
- stdout (normal output)
- stderr (error output)

It assumes `opentrim` is available on your `PATH`. If it is not, set `exe`
to the full path (example in the comment below).

In [9]:
from pathlib import Path
import subprocess
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict

Customize the parameters in the code below:

In [10]:
PARAMETERS: Dict[str, Any] = {
    "Simulation": {
        "simulation_type": "FullCascade",
        "screening_type": "ZBL",
        "electronic_stopping": "SRIM13",
        "electronic_straggling": "Off",
        "nrt_calculation": "NRT_element",
        "intra_cascade_recombination": False,
        "time_ordered_cascades": True,
        "correlated_recombination": True,
        "move_recoil": False,
        "recoil_sub_ed": False
    },
    "Transport": {
        "flight_path_type": "Variable",
        "flight_path_const": 1.0,
        "min_energy": 1.0,
        "min_recoil_energy": 1.0,
        "min_scattering_angle": 2.0,
        "max_rel_eloss": 0.05,
        "mfp_range": [1.0, 1e+30]
    },
    "IonBeam": {
        "ion": {
            "symbol": "H",
            "atomic_number": 1,
            "atomic_mass": 1.00784
        },
        "energy_distribution": {
            "type": "SingleValue",
            "center": 1000000.0,
            "fwhm": 1.0
        },
        "spatial_distribution": {
            "geometry": "Surface",
            "type": "SingleValue",
            "center": [0.0, 5000.0, 5000.0],
            "fwhm": 1.0
        },
        "angular_distribution": {
            "type": "SingleValue",
            "center": [1.0, 0.0, 0.0],
            "fwhm": 1.0
        }
    },
    "Target": {
        "origin": [0.0, 0.0, 0.0],
        "size": [10000.0, 10000.0, 10000.0],
        "cell_count": [100, 1, 1],
        "periodic_bc": [0, 1, 1],
        "materials": [
            {
                "id": "Fe",
                "density": 7.8658,
                "composition": [
                    {
                        "element": {
                            "symbol": "Fe",
                            "atomic_number": 26,
                            "atomic_mass": 55.8452
                        },
                        "X": 1.0,
                        "Ed": 40.0,
                        "El": 3.0,
                        "Es": 3.0,
                        "Er": 40.0,
                        "Rc": 0.946
                    }
                ],
                "color": "#55aaff"
            }
        ],
        "regions": [
            {
                "id": "R1",
                "material_id": "Fe",
                "origin": [0.0, 0.0, 0.0],
                "size": [10000.0, 10000.0, 10000.0]
            }
        ]
    },
    "Output": {
        "title": "1 MeV H on Fe example",
        "outfilename": "Example1",
        "storage_interval": 1000,
        "store_exit_events": True,
        "store_pka_events": True,
        "store_damage_events": False,
        "store_dedx": True
    },
    "Run": {
        "max_no_ions": 20000,
        "max_cpu_time": 0,
        "threads": 4,
        "seed": 123456789
    },
    "UserTally": []
}

# Below are the functions that generate the JSON file:

In [11]:
def build_config(params: Dict[str, Any]) -> Dict[str, Any]:
    """
    Hook to transform/validate parameters before writing.
    For now, it returns params unchanged.
    """
    return params

def write_config(
    params: Dict[str, Any],
    output_path: str | Path = "config.json",
    *,
    pretty: bool = True,
    sort_keys: bool = False,
) -> Path:
    """
    Notebook-friendly writer. Call this after editing `params` (e.g., PARAMETERS).

    Returns the Path to the written file.
    """
    config = build_config(params)

    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with out_path.open("w", encoding="utf-8") as f:
        json.dump(
            config,
            f,
            indent=4 if pretty else None,
            sort_keys=sort_keys,
        )
        f.write("\n")

    return out_path


def config_as_json_string(
    params: Dict[str, Any],
    *,
    pretty: bool = True,
    sort_keys: bool = False,
) -> str:
    """
    Useful in notebooks for display/debugging without writing to disk.
    """
    config = build_config(params)
    return json.dumps(
        config,
        indent=4 if pretty else None,
        sort_keys=sort_keys,
    )
    

out = write_config(PARAMETERS, "config.json", pretty=True)
print("Wrote:", out)

Wrote: config.json


If your PATH is set up correctly, this is enough:

In [12]:
exe = "opentrim"

cmd = [exe, "-f", "config.json", "-o", "result", "-s", "42"]

result = subprocess.run(
    cmd,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    timeout=20,
)

print("returncode:", result.returncode)
print("stdout:\n", result.stdout)
print("stderr:\n", result.stderr)

returncode: 0
stdout:
 Parsing JSON config from config.json
Starting simulation '1 MeV H on Fe example'...


║                                        ║  0%║ETC 00:00:00║
║███████████████▏                        ║ 37%║ETC 00:00:01║
║███████████████████████████████         ║ 77%║ETC 00:00:01║
║████████████████████████████████████████║100%║ETC 00:00:00║
║████████████████████████████████████████║100%║ETC 00:00:00║

Completed 20000 ion histories.
Threads: 4
Cpu time (s):  2.07378,	Ions/cpu-s:  9644.21
Real time (s): 0.607225,	Ions/real-s: 32936.7
Storing results in result.h5 ... OK.

stderr:
 
